# HTGR

## This is an example of a computation of the HTGR that apears in the 9-1 problem of the ANL benchmark book ![](./htgr.png)

### First import all the needed tools

In [ ]:
import inspect

import numpy as np
from IPython.display import Code
from numba.np.unsafe.ndarray import *  # noqa: F403

from dorban.examples import anl_9_htgr
from dorban.settings import FDSettings
from dorban.solve_equation import solve_k
from dorban.utils import space_energy_reshape

## The problem
The materials, the geometry and the core are defined in `dorban.examples.anl_9_htgr`, which the benchmark tests use too:

In [ ]:
Code(inspect.getsource(anl_9_htgr), language="python")

## Defining the core, performing a split and gluing periodic boundary conditions

In [ ]:
system = anl_9_htgr.core()

## Solving for the eigenvalue and the flux, reference eigenvalue is 1.11835

In [ ]:
k, flux = solve_k(system,settings=FDSettings(split=system.geometry.uniform_split(1)))

In [ ]:
print(f"k = {k:.5f}, reference {anl_9_htgr.REFERENCE_K}")

## Computing the fission source

In [ ]:
fission_source = flux*np.hstack(
        [isotope.fission * system.geometry.volumes[cell] for
         cell, isotope in enumerate(system.isotopes)])
fission_source = space_energy_reshape(fission_source,system.geometry.cells,system.E)
fission_source = np.sum(fission_source,axis=1)
non_zero=fission_source.nonzero()
fission_source = fission_source[non_zero]
coef = np.mean(fission_source)/system.geometry.volumes[0]

## Computing the flux in the fuel, fluxes in other regions can be computed similarly.

In [ ]:
group1, group2, group3, group4 = map(lambda x: flux[x::4], range(4))
As=[i for i,iso in enumerate(system.isotopes) if iso.name!="G"]
np.mean(group1[As])/coef,np.mean(group2[As])/coef,np.mean(group3[As])/coef,np.mean(group4[As])/coef